In [ ]:
import pandas as pd
import numpy as np
df_tracks = pd.read_csv('TracksAlmostClean.csv')

In [53]:
#Are there any missing value?
print("df_tracks:",df_tracks.shape)

df_tracks: (11156, 36)


In [6]:
missing_initial_track = df_tracks.isnull().sum() 
# print("The number of missing values is: \n",missing_counts[missing_counts > 0]) #28 columns out of 37
missing_initial_trackdf = missing_initial_track[missing_initial_track > 0].to_frame(name="missing_initial_track_values")
print(missing_initial_trackdf)

          missing_initial_track_values
language                           104
month                              107
day                                130


In [18]:
print("The columns without missing values are: \n", missing_initial_track[missing_initial_track == 0])

The columns without missing values are: 
 id                      0
id_artist               0
title                   0
featured_artists        0
primary_artist          0
album                   0
swear_IT                0
swear_EN                0
swear_IT_words          0
swear_EN_words          0
year                    0
n_sentences             0
n_tokens                0
char_per_tok            0
avg_token_per_clause    0
bpm                     0
rolloff                 0
flux                    0
rms                     0
flatness                0
spectral_complexity     0
pitch                   0
album_name              0
album_release_date      0
album_type              0
disc_number             0
track_number            0
duration_ms             0
explicit                0
popularity              0
id_album                0
lyrics                  0
streams@1month          0
dtype: int64


In [12]:
!pip install langdetect

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     -------------------- ----------------- 524.3/981.5 kB 7.6 MB/s eta 0:00:01
     -------------------------------------- 981.5/981.5 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993250 sha256=66c1ab398fe6561924246014c495a42cca77aae6363f438b32d62a482adfbf2d
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\eb\87\25\2dddf1c94e1786054e25022ec5530bfed52bad86d882999c48
Successfully built langdetect


  DEPRECATION: Building 'langdetect' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'langdetect'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [ ]:
import csv
import re
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException

# --- Configuration ---
# Define the names of your columns
LYRICS_COL = 'lyrics'
LANGUAGE_COL = 'language'
INPUT_FILE = 'TracksAlmostClean.csv'
OUTPUT_FILE = 'output_data_with_language.csv'

def detect_language_safe(text):
    """
    Detects the language of a given text after essential preprocessing.
    """
    if not text or isinstance(text, float):
        return 'undetermined'
    
    # 1. Convert to string and Lowercase
    cleaned_text = str(text).lower()
    
    # 2. Remove symbols, punctuation, and numbers
    # Pattern to keep only letters and spaces (a-z)
    cleaned_text = re.sub(r'[^a-z\s]', '', cleaned_text)
    
    # 3. Clean up extra whitespaces (replaces multiple spaces/tabs with one space)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    # Check if the text is empty after cleaning
    if not cleaned_text:
        return 'undetermined'
        
    try:
        return detect(cleaned_text)
    except LangDetectException:
        return 'undetermined'
    except Exception as e:
        print(f"An unexpected error occurred. Error: {e}")
        return 'error'

# --- Main processing logic using only standard Python types and the csv module ---
def process_lyrics_language(input_filepath, output_filepath):
    """
    Reads a CSV, detects the language for missing values in the language column,
    and writes the results to a new CSV file.
    """
    # This list will hold standard Python dictionaries (the rows)
    processed_rows = [] 
    
    # 1. Read the data
    with open(input_filepath, mode='r', encoding='utf-8', newline='') as infile:
        # DictReader treats rows as standard Python dictionaries
        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames # Get original column headers
        
        # Check column existence (standard Python check)
        if LYRICS_COL not in fieldnames or LANGUAGE_COL not in fieldnames:
            print(f"Error: CSV must contain '{LYRICS_COL}' and '{LANGUAGE_COL}' columns.")
            return

        print("Starting language detection...")
        
        # 2. Process the data row by row
        for row in reader:
            # Check if 'language' is empty (standard string check)
            if not row.get(LANGUAGE_COL, '').strip():
                lyrics = row.get(LYRICS_COL, '')
                
                detected_lang = detect_language_safe(lyrics)
                
                # Directly update the Python dictionary
                row[LANGUAGE_COL] = detected_lang
                
            processed_rows.append(row)
            
    # 3. Write the results
    with open(output_filepath, mode='w', encoding='utf-8', newline='') as outfile:
        # DictWriter writes standard Python dictionaries
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        
        writer.writeheader()
        writer.writerows(processed_rows)
        
    print(f"\nProcessing complete! Results saved to '{output_filepath}'")

In [15]:
process_lyrics_language(INPUT_FILE, OUTPUT_FILE)

Starting language detection...

Processing complete! Results saved to 'output_data_with_language.csv'


In [16]:
dff = pd.read_csv('output_data_with_language.csv')
missing_second_track = dff.isnull().sum() 
# print("The number of missing values is: \n",missing_counts[missing_counts > 0]) #28 columns out of 37
missing_second_trackdf = missing_second_track[missing_second_track > 0].to_frame(name="missing_second_track_values")
print(missing_second_trackdf)


       missing_second_track_values
month                          107
day                            130


In [17]:
dff["language"].unique()

array(['pl', 'en', 'it', 'fr', 'eu', 'sco', 'co', 'da', 'nl', 'et', 'pt',
       'es', 'war', 'lt', 'ia', 'de', 'cs', 'gl', 'sr', 'rw', 'ro', 'ca',
       'no', 'sk', 'aa', 'chr', 'rm', 'ru', 'mt', 'qu', 'cy', 'eo', 'bg',
       'la', 'sq', 'sw'], dtype=object)

In [27]:
dff['album_release_date'].head(15)

0     2021-04-09
1     2021-04-09
2     2021-04-09
3     2025-05-16
4     2020-05-28
5     2020-05-28
6     2020-05-28
7     2020-05-28
8     2023-02-09
9     2020-05-28
10    2022-06-02
11    2020-05-28
12    2020-05-28
13    2020-05-28
14    2021-04-09
Name: album_release_date, dtype: object

In [29]:
date_parsed = pd.to_datetime(dff['album_release_date'], format='%Y-%m-%d', errors='coerce')

# Righe che NON rispettano il formato yyyy-mm-dd
non_valide = dff[date_parsed.isna()]

print(non_valide[['album_release_date']].head())


     album_release_date
283                2013
443                1955
1101               2010
1113               2010
1547               2004


In [30]:
date_parsed = pd.to_datetime(dff['album_release_date'], format='%Y-%m-%d', errors='coerce')

#Remove the invalid rows
df_clean = dff[date_parsed.notna()].copy()

In [85]:
df_clean

,id,id_artist,title,featured_artists,primary_artist,language,album,swear_IT,swear_EN,swear_IT_words,...,album_type,disc_number,track_number,duration_ms,explicit,popularity,id_album,lyrics,streams@1month,season
0,TR934808,ART04205421,​polka 2 :-/,"Ernia, Guè",Rosa Chemical,pl,FOREVER AND EVER,13,6,"['cazzo', 'cesso', 'coglioni', 'figa', 'merda'...",...,album,1.0,3.0,207761.0,True,46.0,ALB115557,"Oplà, ah\r\nBdope, chiama due b—\r\n\r\nMi can...",186522.0,spring
1,TR760029,ART04205421,POLKA,Thelonious B.,Rosa Chemical,en,FOREVER AND EVER,9,12,"['cazzo', 'culo', 'frocio', 'puttana', 'sega',...",...,album,1.0,3.0,207761.0,True,46.0,ALB115557,"Greg Willen, non dormire\r\n(Brr-poh)\r\n\r\nT...",194313.0,spring
2,TR916821,ART04205421,​britney ;-),"MamboLosco, RADICAL",Rosa Chemical,en,FOREVER AND EVER,16,12,"['bastardo', 'cazzo', 'culo', 'merda', 'troia']",...,album,1.0,1.0,193544.0,True,39.0,ALB115557,"Mothz\r\nYeah, yeah, yeah-yeah\r\nBdope, chiam...",63750.0,winter
3,TR480968,ART04205421,CEO,Taxi B,Rosa Chemical,it,OKAY OKAY !! - EP,8,3,"['cazzo', 'culo', 'fottere', 'merda', 'pompino...",...,single,1.0,2.0,169000.0,True,47.0,ALB730959,Designer sui vestiti penso di essere un outlet...,41473.0,spring
4,TR585039,ART04205421,LONDRA,Rkomi,Rosa Chemical,en,FOREVER AND EVER,1,0,['cazzo'],...,album,1.0,8.0,194779.0,True,41.0,ALB436151,"Bdope (Yeah)\r\n\r\nVuole solo me, non fare la...",30553.0,spring
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11151,TR434449,ART02733420,Badabum Cha Cha - Gabri Ponte & Paki Rmx (Radi...,No Feature,Marracash,it,Badabum Cha Cha (The Remixes),0,0,[],...,single,1.0,1.0,234374.0,False,57.0,ALB248347,"Qui non va, ma questo badabum cha cha\r\nTira ...",4663.0,summer
11152,TR826351,ART02733420,Senza Un Posto Nel Mondo - Alternative Version,Tiziano Ferro,Marracash,it,Senza Un Posto Nel Mondo – Single,1,0,['fortuna'],...,album,1.0,20.0,248330.0,True,24.0,ALB100713,Odio questa città (città)\r\nSappiamo che la v...,5686.0,autumn
11153,TR529809,ART02733420,Badabum Cha Cha - Gabri Ponte & Paki Rmx (Exte...,No Feature,Marracash,it,Badabum Cha Cha (The Remixes),0,0,[],...,single,1.0,1.0,234374.0,False,57.0,ALB248347,"Qui non va, ma questo badabum cha cha\r\nTira ...",5540.0,summer
11154,TR280904,ART02733420,Tempo,No Feature,Marracash,it,Status,0,1,[],...,album,1.0,7.0,302173.0,True,37.0,ALB100713,É crudo il suono e questa vibra che ti do\r\nC...,4672.0,winter


In [33]:
df_clean[df_clean['day'].isna() & df_clean['month'].isna()][['id', 'title', 'album_release_date', 'day', 'month']]

,id,title,album_release_date,day,month


In [39]:
df_clean.isnull().sum() 


id                      0
id_artist               0
title                   0
featured_artists        0
primary_artist          0
language                0
album                   0
swear_IT                0
swear_EN                0
swear_IT_words          0
swear_EN_words          0
year                    0
month                   0
day                     0
n_sentences             0
n_tokens                0
char_per_tok            0
avg_token_per_clause    0
bpm                     0
rolloff                 0
flux                    0
rms                     0
flatness                0
spectral_complexity     0
pitch                   0
album_name              0
album_release_date      0
album_type              0
disc_number             0
track_number            0
duration_ms             0
explicit                0
popularity              0
id_album                0
lyrics                  0
streams@1month          0
dtype: int64

In [69]:
sorted(df_clean["day"].unique())
df_clean["day"] = df_clean["day"].astype(int)

In [80]:
sorted(df_clean["year"].unique())
df_clean["year"] = df_clean["year"].astype(int)

In [42]:
sorted(df_clean["month"].unique())

[np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(11),
 np.int64(12)]

In [ ]:
#month was a float, now we trasform it to int
df_clean["month"] = df_clean["month"].astype(int)


In [43]:
#create a new column with the season
def month_to_season(month):
    if pd.isna(month):
        return "Unknown"
    month = int(month)
    if month in (12, 1, 2):
        return "Winter"
    elif month in (3, 4, 5):
        return "Spring"
    elif month in (6, 7, 8):
        return "Summer"
    elif month in (9, 10, 11):
        return "Autumn"
    else:
        return "Unknown"


In [44]:
df_clean["season"] = df_clean["month"].apply(month_to_season)

In [49]:
df_clean["season"] = df_clean["season"].str.lower()

In [50]:
df_clean["season"].unique()

array(['spring', 'winter', 'summer', 'autumn'], dtype=object)

In [51]:
df_clean["season"].value_counts()

season
spring    3428
autumn    2827
winter    2619
summer    2021
Name: count, dtype: int64

In [57]:
df_clean.columns

Index(['id', 'id_artist', 'title', 'featured_artists', 'primary_artist',
       'language', 'album', 'swear_IT', 'swear_EN', 'swear_IT_words',
       'swear_EN_words', 'year', 'month', 'day', 'n_sentences', 'n_tokens',
       'char_per_tok', 'avg_token_per_clause', 'bpm', 'rolloff', 'flux', 'rms',
       'flatness', 'spectral_complexity', 'pitch', 'album_name',
       'album_release_date', 'album_type', 'disc_number', 'track_number',
       'duration_ms', 'explicit', 'popularity', 'id_album', 'lyrics',
       'streams@1month', 'season'],
      dtype='object')

In [73]:
#repeat both for year,month,day
df_clean['day'].min() 
df_clean['day'].max() 

31

In [ ]:
#year has a min=1992 e max=2025 -->ok
#month has a min=1 e max=12 -->ok
#month has a min=1 e max=31 -->ok

In [ ]:
# Max day in each month
max_day_per_month = df_clean.groupby('month')['day'].max()

print(max_day_per_month)

month
1     31
2     31
3     31
4     30
5     31
6     30
7     31
8     31
9     30
10    31
11    30
12    31
Name: day, dtype: int64


In [81]:
#notice an error in february (month=2) with day=30
# Filtra righe di febbraio con day > 28
febbraio_anomale = df_clean[(df_clean['month'] == 2) & (df_clean['day'] > 28)]

# Seleziona solo le colonne desiderate
febbraio_anomale = febbraio_anomale[["id", "primary_artist", "album_release_date", "day", "month","year", "season"]]

print(febbraio_anomale)

             id     primary_artist album_release_date  day  month  year  \
2436   TR761814           Mistaman         2012-02-22   29      2  2012   
2471   TR971470           Mistaman         2012-02-22   29      2  2012   
2478   TR888527           Mistaman         2012-02-22   29      2  2012   
2479   TR901922           Mistaman         2012-02-22   29      2  2012   
2480   TR981030           Mistaman         2008-07-29   29      2  2012   
2486   TR888249           Mistaman         2012-02-22   29      2  2012   
2490   TR992688           Mistaman         2012-02-22   29      2  2012   
2493   TR918231           Mistaman         2012-02-22   29      2  2012   
3699  TR_379799  Frankie hi-nrg mc         2008-02-29   29      2  2008   
3707  TR_209885  Frankie hi-nrg mc         2008-02-29   29      2  2008   
3709  TR_714723  Frankie hi-nrg mc         2008-02-29   29      2  2008   
3711  TR_178795  Frankie hi-nrg mc         2008-02-29   29      2  2008   
3721  TR_751449  Frankie 

In [ ]:
#convert "album_release_date" to datetime and check for inconsistencies
df_clean['album_release_date'] = pd.to_datetime(df_clean['album_release_date'], errors='coerce')

# crate a new boolean column to check if year, month, day match with album_release_date
df_clean['date_matches'] = (
    (df_clean['year'] == df_clean['album_release_date'].dt.year) &
    (df_clean['month'] == df_clean['album_release_date'].dt.month) &
    (df_clean['day'] == df_clean['album_release_date'].dt.day)
)

# flilter rows where date does not match
incoerenti = df_clean[df_clean['date_matches'] == False]

# Visualization of inconsistent rows
incoerenti = incoerenti[["id", "primary_artist", "album_release_date", "year", "month", "day", "season"]]

print(incoerenti)


             id primary_artist album_release_date  year  month  day  season
0      TR934808  Rosa Chemical         2021-04-09  2021      4    2  spring
1      TR760029  Rosa Chemical         2021-04-09  2020      3    6  spring
2      TR916821  Rosa Chemical         2021-04-09  2021      2   19  winter
3      TR480968  Rosa Chemical         2025-05-16  2019      3    8  spring
4      TR585039  Rosa Chemical         2020-05-28  2020      5   29  spring
...         ...            ...                ...   ...    ...  ...     ...
11145  TR479545      Marracash         2014-12-17  2019     10   29  autumn
11146  TR153241      Marracash         2014-09-23  2014      1    7  winter
11147  TR216791      Marracash         2018-12-07  2019     10   21  autumn
11152  TR826351      Marracash         2015-02-10  2015     11    9  autumn
11154  TR280904      Marracash         2015-02-10  2005      1    1  winter

[6087 rows x 7 columns]


In [87]:
# 1. Identify Inconsistent Rows
# The inconsistent_mask is a boolean Series where True indicates a mismatch
# between the date components (year, month, day) and the parsed datetime object.
inconsistent_mask = (df_clean['date_matches'] == False)

# 2. Resolve Inconsistencies by updating 'year', 'month', and 'day' columns
# The strategy is to overwrite the inconsistent values with the correct values
# extracted from the 'album_release_date' datetime column.

# Crucially, this method handles both valid and missing dates:
# - If 'album_release_date' is a valid datetime, it extracts the correct component.
# - If 'album_release_date' is NaT (Not a Time, i.e., invalid), .dt.year/.dt.month/.dt.day
#   will return NaN, thus correctly setting the component columns to missing.

# Update 'year' column for inconsistent rows
df_clean.loc[inconsistent_mask, 'year'] = df_clean.loc[inconsistent_mask, 'album_release_date'].dt.year

# Update 'month' column for inconsistent rows
df_clean.loc[inconsistent_mask, 'month'] = df_clean.loc[inconsistent_mask, 'album_release_date'].dt.month

# Update 'day' column for inconsistent rows
df_clean.loc[inconsistent_mask, 'day'] = df_clean.loc[inconsistent_mask, 'album_release_date'].dt.day

# --- Verification (Recommended) ---

# 3. Recalculate 'date_matches' to confirm the fix
df_clean['date_matches'] = (
    (df_clean['year'] == df_clean['album_release_date'].dt.year) &
    (df_clean['month'] == df_clean['album_release_date'].dt.month) &
    (df_clean['day'] == df_clean['album_release_date'].dt.day)
)

# 4. Print results to confirm resolution
remaining_inconsistencies = (~df_clean['date_matches']).sum()

print("----------------------------------------------------------------------")
print("Date Inconsistencies Resolved by aligning separate columns with 'album_release_date'.")
print(f"Remaining rows where 'date_matches' is False: {remaining_inconsistencies}")
print("----------------------------------------------------------------------")

----------------------------------------------------------------------
Date Inconsistencies Resolved by aligning separate columns with 'album_release_date'.
Remaining rows where 'date_matches' is False: 0
----------------------------------------------------------------------


In [96]:
# List of valid leap years for the date 02/29
valid_leap_years = [1992, 1996, 2000, 2004, 2008, 2012, 2016, 2020, 2024]

# 1. Create a mask for rows that are 'February 29th'
feb_29_mask = (df_clean['month'] == 2) & (df_clean['day'] == 29)

# 2. Identify the anomalies: February 29th but year is NOT in the valid list
anomalous_leap_rows = df_clean[feb_29_mask & (~df_clean['year'].isin(valid_leap_years))]

# --- Visualizzazione ---

# Count the anomalies
anomaly_count = len(anomalous_leap_rows)

print("--- Check on 29 February ---")
print(f"Number of row with date non accetable: {anomaly_count}")

# 3. Print the anomalous rows
if anomaly_count > 0:
    print("\n--- Anomalous rows ---")
    
    # Visualization of the key date columns for the anomalies
    print(anomalous_leap_rows[[
        "album_release_date", 
        "year", 
        "month", 
        "day"
    ]])
else:
    print("\nPerfect! All date of February 29th' are in a leap year.")

--- Check on 29 February ---
Number of row with date non accetable: 0

Perfect! All date of February 29th' are in a leap year.


In [92]:
df_clean["date_matches"].unique()

array([ True])

In [93]:
df_clean['album_release_date'].dtype

dtype('<M8[ns]')

In [94]:
#now we can drop the column date_matches
df_clean = df_clean.drop(columns=['date_matches'])

In [99]:
#check for duplicates (already done and there are none)
# in particular on the id column
print(df_clean[df_clean.duplicated(subset=['id'], keep=False)])

             id    id_artist                                       title  \
43     TR715264  ART04205421                              ​non è normale   
120    TR976686  ART19605256                                      Ibridi   
141    TR230274  ART18853907                                 SaN LoREnZo   
159    TR531651  ART18853907  Serenata - From “Forever Out of My League”   
199    TR898853  ART88026810                                     ​oh 9od   
...         ...          ...                                         ...   
10869  TR292480  ART07024718                             Nel mio piccolo   
10905  TR978886  ART07024718                      L’arte di accontentare   
10952  TR925275  ART07024718                              Bimbo Speciale   
10997  TR747430  ART02733420                                  Scooteroni   
11150  TR458543  ART02733420             Di Nascosto (Don Joe Demo 2014)   

      featured_artists primary_artist language  \
43          No Feature  Rosa Chemical

In [ ]:
# 1. filter to check ALL rows involved in duplicates on 'id' column
df_duplicates = df_clean[df_clean.duplicated(subset=['id'], keep=False)]

# 2. Count hpw many times each 'id' appears among the duplicates
count_by_id = df_duplicates['id'].value_counts()

# 3. Print
print(count_by_id)

id
TR367132    4
TR715264    2
TR230274    2
TR976686    2
TR531651    2
           ..
TR247772    2
TR261964    2
TR386339    2
TR825208    2
TR997536    2
Name: count, Length: 68, dtype: int64


In [ ]:
# only for id "TR367132" we have dublicate because one is a single and one in the album
# all the others are completely different rows

In [ ]:
# group by id'and identify groups with different "primary_artist" o "title".
artist_counts = df_clean.groupby('id')['primary_artist'].nunique()
title_counts = df_clean.groupby('id')['title'].nunique()

# rename id with different artist or title
broken_ids_by_artist = artist_counts[artist_counts > 1].index.tolist()
broken_ids_by_title = title_counts[title_counts > 1].index.tolist()
broken_ids_to_rename = list(set(broken_ids_by_artist + broken_ids_by_title))

print(f"ID to rename for errors: {broken_ids_to_rename}")

ID da rinominare per errore di assegnazione: ['TR898853', 'TR976686', 'TR464052', 'TR362754', 'TR503521', 'TR475283', 'TR621186', 'TR726762', 'TR731836', 'TR737119', 'TR798937', 'TR323298', 'TR716382', 'TR531651', 'TR292480', 'TR135764', 'TR743448', 'TR772702', 'TR809682', 'TR497887', 'TR922726', 'TR261964', 'TR938316', 'TR230274', 'TR383581', 'TR237380', 'TR825208', 'TR997536', 'TR324280', 'TR388228', 'TR192351', 'TR747430', 'TR903275', 'TR970205', 'TR282769', 'TR978886', 'TR638268', 'TR761888', 'TR367132', 'TR423710', 'TR213881', 'TR517220', 'TR690925', 'TR866344', 'TR458543', 'TR896087', 'TR108862', 'TR679972', 'TR987615', 'TR247772', 'TR190585', 'TR869063', 'TR925275', 'TR205970', 'TR386339', 'TR318448', 'TR245683', 'TR371139', 'TR597980', 'TR420065', 'TR577214', 'TR844214', 'TR715264', 'TR980497', 'TR754440', 'TR448308', 'TR757765', 'TR534818']


In [116]:
len(broken_ids_to_rename)

68

In [ ]:
for broken_id in broken_ids_to_rename:
    # find the indexes of all rows with that ID
    group_indices = df_clean[df_clean['id'] == broken_id].index

    #Rename the exceeding occurrences beyond the first
    for i in range(1, len(group_indices)):
        new_id_str = f"TR{current_new_id:06.0f}" # new ID with TR prefix
        
        # Assign the new ID to the 'id' column
        df_clean.loc[group_indices[i], 'id'] = new_id_str
        
        # update the counter
        current_new_id += 1
        
print("Rename completed.")

Riassegnazione degli ID errati completata.


In [ ]:
# check again for duplicates and there are none
print(df_clean[df_clean.duplicated(subset=['id'], keep=False)])

Empty DataFrame
Columns: [id, id_artist, title, featured_artists, primary_artist, language, album, swear_IT, swear_EN, swear_IT_words, swear_EN_words, year, month, day, n_sentences, n_tokens, char_per_tok, avg_token_per_clause, bpm, rolloff, flux, rms, flatness, spectral_complexity, pitch, album_name, album_release_date, album_type, disc_number, track_number, duration_ms, explicit, popularity, id_album, lyrics, streams@1month, season, id_numeric]
Index: []

[0 rows x 38 columns]


In [120]:
#now we can drop the column id_numeric
df_clean = df_clean.drop(columns=['id_numeric'])

In [ ]:
# Check if primary_artist appears in featured_artists (without creating a new column)
primary_featured_issues = df_clean[
    df_clean.apply(
        lambda row: row['primary_artist'] in row['featured_artists'].split(',') 
        if pd.notna(row['featured_artists']) else False,
        axis=1
    )
]

print("Rows where primary_artist appears in featured_artists:")
print(primary_featured_issues[['id', 'primary_artist', 'featured_artists']])

Rows where primary_artist appears in featured_artists:
Empty DataFrame
Columns: [id, primary_artist, featured_artists]
Index: []


In [124]:
# Check album consistency per id_album
album_consistency = df_clean.groupby('id_album').agg({
    'album_name': pd.Series.nunique,
    'album_type': pd.Series.nunique
}).reset_index()

inconsistent_albums = album_consistency[
    (album_consistency['album_name'] > 1) | (album_consistency['album_type'] > 1)
]

print("\nAlbums with inconsistent names or types per id_album:")
print(inconsistent_albums)


Albums with inconsistent names or types per id_album:
       id_album  album_name  album_type
180   ALB156429           2           1
361   ALB213075           1           2
578   ALB275546           2           1
1227  ALB473208           2           2
1330  ALB504235           2           2
2223  ALB774561           1           2
2501  ALB858106           1           2
2582  ALB877400           2           2


In [142]:
for album_id in inconsistent_albums['id_album']:
    album_rows = df_clean[df_clean['id_album'] == album_id]
    # Controlla se ci sono più artisti
    if album_rows['primary_artist'].nunique() > 1:
        print(f"Album {album_id} has multiple artists: manual check needed")
    else:
        # Stesso artista → uniforma album_name e album_type
        most_common_name = album_rows['album_name'].mode()[0]
        most_common_type = album_rows['album_type'].mode()[0]
        df_clean.loc[df_clean['id_album'] == album_id, 'album_name'] = most_common_name
        df_clean.loc[df_clean['id_album'] == album_id, 'album_type'] = most_common_type


Album ALB156429 has multiple artists: manual check needed
Album ALB275546 has multiple artists: manual check needed
Album ALB473208 has multiple artists: manual check needed
Album ALB504235 has multiple artists: manual check needed
Album ALB877400 has multiple artists: manual check needed


In [146]:
df_clean["id_album"].unique()

array(['ALB115557', 'ALB730959', 'ALB436151', ..., 'ALB503262',
       'ALB937776', 'ALB248347'], dtype=object)

In [ ]:
album_ids_array = df_clean['id_album'].unique()
prefix = "ALB"

# check if all album IDs start with "ALB"
all_start_with_alb = all(str(x).startswith(prefix) for x in album_ids_array)

print(f"Does all the id_album start with '{prefix}'? {all_start_with_alb}")

Tutti gli ID degli album iniziano con 'ALB'? False


In [ ]:
import pandas as pd

#obtain unique album IDs
album_ids_array = df_clean['id_album'].unique()
prefix = "ALB"

# identify non-conforming album IDs
non_alb_ids = [
    x for x in album_ids_array 
    if pd.notna(x) and not str(x).startswith(prefix)
]

# calculate the number of non-conforming IDs
count_non_alb = len(non_alb_ids)

# print
print(f"Number of id_album NOT statring with '{prefix}': {count_non_alb}")

if count_non_alb > 0:
    print("\nWrong id_album (first 10):")
    print(non_alb_ids[:10])

Numero di ID album che NON iniziano con 'ALB': 12

Esempi di ID non conformi (primi 10):
['52klMHkvfphE8PiD5ziyMP', '4aaVFa8RcMt5M2q7fr7wyi', '1iMtC9ZMR2zqC74aNP510k', '0V7I5biUrCTpWe4uxTWd69', '4uqe7ExJ4JKoxesfnpPkpd', '3DpcsZhaN5zpb7qnHniReD', '2fAvKgNnIbaNWsGJTwd3k2', '2mhQHkfig02lfxjHBOoo1q', '5rgklJ2KFv2ROyyNaRiuEy', '2gNnGtEgJvPSulctj6zCZF']


In [ ]:
# clean the df by removing rows with non conforming album ids
rows_before = df_clean.shape[0]

# filter the DataFrame to exclude rows whose 'id_album' is in the non conforming list
df_clean_filtered = df_clean[~df_clean['id_album'].isin(non_alb_ids)].copy()

rows_after = df_clean_filtered.shape[0]

print(f"Rows in the original dataset: {rows_before}")
print(f"Removed rows: {rows_before - rows_after}")
print(f"Rows in the filter dataset: {rows_after}")

# subtitute the old DataFrame with the filtered one
df_clean = df_clean_filtered

Righe nel DataFrame originale: 10895
Righe rimosse: 71
Righe nel nuovo DataFrame filtrato: 10824


In [152]:
#5 out of 8 need a rename
# Find the max numeric part of existing ALB IDs
existing_ids = df_clean['id_album'].str.extract(r'ALB(\d+)')[0].astype(int)
current_new_id = existing_ids.max() + 1

# Step 3: Process each inconsistent album
for album_id in inconsistent_albums['id_album']:
    album_rows = df_clean[df_clean['id_album'] == album_id]

    # Case A: multiple artists → assign new IDs to all but the first occurrence
    if album_rows['primary_artist'].nunique() > 1:
        group_indices = album_rows.index
        # Keep the first row, rename the others
        for i in range(1, len(group_indices)):
            new_id_str = f"ALB{current_new_id:06d}"
            df_clean.loc[group_indices[i], 'id_album'] = new_id_str
            current_new_id += 1

print("Album inconsistencies processed.")

Album inconsistencies processed.


In [155]:
# --- 1. Define columns and limits ---
text_features = ['n_sentences', 'n_tokens', 'char_per_tok', 'avg_token_per_clause']
audio_features = ['bpm', 'rolloff', 'flux', 'rms', 'flatness', 'spectral_complexity', 'pitch']
duration_col = 'duration_ms'

# Physical/plausible limits for audio features
audio_limits = {
    'bpm': (20, 300),         # beats per minute should be 20-300
    'pitch': (0, None),       # pitch cannot be negative
    # Other audio features: general negative check
}

# Duration limits: 10 sec to 20 min in ms
duration_limits = (10000, 20*60*1000)

# --- 2. Create an empty list to collect issues ---
issues = []

# --- 3. Check text features ---
for col in text_features:
    invalid = df_clean.loc[df_clean[col] <= 0]
    for idx, row in invalid.iterrows():
        issues.append({
            'id': row['id'],
            'column': col,
            'value': row[col],
            'issue': '<= 0'
        })

# --- 4. Check audio features ---
for col in audio_features:
    if col in audio_limits:
        low, high = audio_limits[col]
        # Create condition aligned with DataFrame index
        condition = pd.Series(False, index=df_clean.index)
        if low is not None:
            condition |= df_clean[col] < low
        if high is not None:
            condition |= df_clean[col] > high
        invalid = df_clean.loc[condition]
    else:
        # General check for negative values
        invalid = df_clean.loc[df_clean[col] < 0]
    
    for idx, row in invalid.iterrows():
        issue_desc = 'out of range' if col in audio_limits else 'negative'
        issues.append({
            'id': row['id'],
            'column': col,
            'value': row[col],
            'issue': issue_desc
        })

# --- 5. Check duration ---
invalid_duration = df_clean.loc[
    (df_clean[duration_col] < duration_limits[0]) |
    (df_clean[duration_col] > duration_limits[1])
]
for idx, row in invalid_duration.iterrows():
    issues.append({
        'id': row['id'],
        'column': duration_col,
        'value': row[duration_col],
        'issue': 'too short or too long'
    })

# --- 6. Convert issues to a DataFrame ---
issues_df = pd.DataFrame(issues)

# --- 7. Summary ---
print(f"Total suspicious values found: {len(issues_df)}")
print(issues_df.head(20))  # Show first 20 issues for review

# Optional: save to CSV for auditing
# issues_df.to_csv("numeric_issues_report.csv", index=False)

print("Complete numeric check finished.")

Total suspicious values found: 118
          id                column  value issue
0   TR440692  avg_token_per_clause    0.0  <= 0
1   TR406572  avg_token_per_clause    0.0  <= 0
2   TR568257  avg_token_per_clause    0.0  <= 0
3   TR418448  avg_token_per_clause    0.0  <= 0
4   TR480687  avg_token_per_clause    0.0  <= 0
5   TR735016  avg_token_per_clause    0.0  <= 0
6   TR750777  avg_token_per_clause    0.0  <= 0
7   TR546120  avg_token_per_clause    0.0  <= 0
8   TR502373  avg_token_per_clause    0.0  <= 0
9   TR875471  avg_token_per_clause    0.0  <= 0
10  TR425349  avg_token_per_clause    0.0  <= 0
11  TR248876  avg_token_per_clause    0.0  <= 0
12  TR480925  avg_token_per_clause    0.0  <= 0
13  TR408042  avg_token_per_clause    0.0  <= 0
14  TR128736  avg_token_per_clause    0.0  <= 0
15  TR865167  avg_token_per_clause    0.0  <= 0
16  TR270467  avg_token_per_clause    0.0  <= 0
17  TR833403  avg_token_per_clause    0.0  <= 0
18  TR890254  avg_token_per_clause    0.0  <= 0
19  T

In [157]:
df_clean.loc[df_clean['avg_token_per_clause'] == 0, ['id', 'lyrics', 'n_tokens', 'n_sentences']]

,id,lyrics,n_tokens,n_sentences
58,TR440692,1 ContributorLATTE + Lyrics,4.0,1.0
61,TR406572,2 ContributorsPunk ’a Piana Lyrics,6.0,1.0
65,TR568257,1 ContributorC. A. S. Lyrics,5.0,2.0
68,TR418448,1 ContributorLuciano Pavarotty // Glock Lyrics,7.0,1.0
262,TR480687,"\r\nOh-oh-oh\r\n\r\nYeah, yeah-eh\r\nFumo, fum...",130.0,16.0
...,...,...,...,...
10324,TR686662,2 ContributorsIl peggior night Lyrics,5.0,1.0
10639,TR177607,2 ContributorsIntro MDSK Lyrics,4.0,1.0
10643,TR749503,2 ContributorsOutro LyricsThis song is an inst...,7.0,1.0
10674,TR579061,1 ContributorPrimo interludio Lyrics,4.0,1.0


In [61]:
df_clean.describe()

,swear_IT,swear_EN,year,month,day,n_sentences,n_tokens,char_per_tok,avg_token_per_clause,bpm,...,flux,rms,flatness,spectral_complexity,pitch,disc_number,track_number,duration_ms,popularity,streams@1month
count,10895.000000,10895.000000,10895.000000,10895.000000,10895.000000,10895.000000,10895.000000,10895.000000,10895.000000,10895.000000,...,10895.000000,10895.000000,10895.000000,10895.000000,10895.000000,10895.000000,10895.000000,1.089500e+04,10895.000000,1.089500e+04
mean,2.347774,0.718587,2015.351721,6.130335,15.338045,59.343552,496.211932,4.053473,8.049039,114.207911,...,1.258442,0.224547,0.860420,27.441367,2255.949281,1.016613,6.835062,2.028881e+05,32.817164,1.848484e+04
std,3.702832,2.567988,6.924043,3.488986,9.098920,24.628205,208.211326,0.427538,14.720561,26.714487,...,0.136586,0.064277,0.107605,8.387582,380.932535,0.138838,5.194550,8.852878e+04,19.801766,3.944815e+04
min,0.000000,0.000000,1992.000000,1.000000,1.000000,1.000000,3.000000,2.000000,0.000000,59.970000,...,0.000000,0.000000,0.109400,0.000000,0.000000,1.000000,1.000000,1.142600e+04,0.000000,4.011000e+03
25%,0.000000,0.000000,2011.000000,3.000000,8.000000,46.000000,372.000000,3.867982,5.870968,91.950000,...,1.172050,0.187400,0.841700,21.974000,2004.899000,1.000000,2.000000,1.696755e+05,16.000000,4.840000e+03
50%,1.000000,0.000000,2017.000000,6.000000,15.000000,58.000000,491.000000,4.013825,6.775862,107.010000,...,1.257600,0.229700,0.882400,27.397400,2244.537000,1.000000,6.000000,1.961620e+05,32.000000,5.662000e+03
75%,3.000000,0.000000,2021.000000,9.000000,23.000000,73.000000,613.000000,4.169753,8.104564,134.190000,...,1.345400,0.267700,0.912800,32.916400,2489.748800,1.000000,10.000000,2.268675e+05,47.000000,1.459450e+04
max,72.000000,72.000000,2025.000000,12.000000,31.000000,437.000000,3089.000000,10.666667,660.000000,738.270000,...,1.928500,0.621900,1.000000,61.222500,3993.020300,5.000000,54.000000,3.753057e+06,100.000000,1.501925e+06


In [160]:
# Esporta
df_clean.to_csv("output_artist_clean.csv", index=False)

print("File created: output_artist_clean.csv")

File created: output_artist_clean.csv
